# Dia 11 — Decorators & Context Managers para Data Science

**Autor:** Gutemberg Rapôso · [GitHub](https://github.com/Gutembergfox) · Palmas-TO, Brasil
**Fase 1 — Python Basics** · Dia 11/28 · 24/10/2026

## Introdução

**Decorators** são funções que recebem outra função e devolvem uma versão "embrulhada" dela,
adicionando comportamento (medir tempo, registrar logs, validar entradas, tentar novamente)
**sem alterar o código original**. **Context managers** (o bloco `with`) garantem que recursos
sejam abertos e **sempre** liberados — mesmo quando ocorre um erro.

### Por que isso é essencial para um cientista de dados profissional?

- **Reprodutibilidade e observabilidade:** medir tempo e registrar chamadas de cada etapa do pipeline.
- **Robustez:** validar DataFrames antes de processar e repetir automaticamente cargas instáveis.
- **Desempenho:** memoizar cálculos caros com `lru_cache`.
- **Segurança de recursos:** conexões de banco, arquivos temporários e opções do pandas restaurados automaticamente.
- **Código limpo:** separar a *lógica de negócio* das *preocupações transversais* (logging, timing, validação).

**Dados utilizados (somente públicos e reais):** `tips`, `titanic`, `iris` e `penguins` do seaborn,
e o texto das descrições (`DESCR`) dos datasets embutidos do scikit-learn como pequeno corpus para NLP.

In [1]:
# Core imports used throughout the notebook
import contextlib
import functools
import logging
import os
import shutil
import sqlite3
import sys
import tempfile
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import datasets as skdatasets

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | seaborn {sns.__version__}")

Python 3.11.17 | pandas 2.3.3 | seaborn 0.13.2


In [2]:
# Configure a notebook-friendly logger (writes to stdout so output is captured by nbconvert)
logger = logging.getLogger("ds_portfolio")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
logger.addHandler(_handler)
logger.propagate = False
logger.info("Logger ready")

[INFO] Logger ready


---
# Parte 1 — Python Decorators

## 1.1 Conceito básico: funções são objetos

Em Python, funções são **objetos de primeira classe**: podem ser atribuídas a variáveis, passadas
como argumento e retornadas por outras funções. Um decorator explora exatamente isso:

```python
@decorator
def f(): ...
# é equivalente a
f = decorator(f)
```

In [3]:
def shout(func):
    """Minimal decorator: wraps `func` and upper-cases its string result."""
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper


def greet(name):
    """Return a friendly greeting."""
    return f"hello, {name}"

# Manual decoration (no @ syntax) to make the mechanism explicit
greet_loud = shout(greet)
print(greet("data scientist"))
print(greet_loud("data scientist"))

hello, data scientist
HELLO, DATA SCIENTIST


In [4]:
@shout
def farewell(name):
    """Return a farewell message."""
    return f"goodbye, {name}"

print(farewell("pandas"))
# Problem: metadata of the original function is lost!
print("__name__:", farewell.__name__)
print("__doc__ :", farewell.__doc__)

GOODBYE, PANDAS
__name__: wrapper
__doc__ : None


## 1.2 `functools.wraps` — preservando metadados

Repare que `farewell.__name__` virou `wrapper` e a docstring sumiu. Isso atrapalha a depuração,
a documentação automática (`help()`, Sphinx) e ferramentas como o pytest. A solução é usar
`@functools.wraps(func)` dentro do decorator — ele copia `__name__`, `__doc__`, `__module__`
e expõe a função original em `__wrapped__`.

In [5]:
def shout_v2(func):
    """Upper-case decorator that preserves the wrapped function's metadata."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper


@shout_v2
def farewell_v2(name):
    """Return a farewell message."""
    return f"goodbye, {name}"

print(farewell_v2("seaborn"))
print("__name__   :", farewell_v2.__name__)
print("__doc__    :", farewell_v2.__doc__)
print("__wrapped__:", farewell_v2.__wrapped__("original, undecorated"))

GOODBYE, SEABORN
__name__   : farewell_v2
__doc__    : Return a farewell message.
__wrapped__: goodbye, original, undecorated


## 1.3 Decorator `@timer` — medindo tempo de execução

Em DS, saber **quanto tempo cada etapa leva** é fundamental para identificar gargalos.
Usamos `time.perf_counter()`, o relógio de maior resolução para medir intervalos.
Também guardamos o último tempo em um atributo da função (`wrapper.last_elapsed`) para
poder analisá-lo depois.

In [6]:
def timer(func):
    """Measure and log the wall-clock execution time of `func`.

    The last measured duration (in seconds) is stored in `wrapper.last_elapsed`.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            wrapper.last_elapsed = time.perf_counter() - start
            logger.info(f"@timer  {func.__name__} took {wrapper.last_elapsed * 1000:.2f} ms")
    wrapper.last_elapsed = None
    return wrapper

In [7]:
tips = sns.load_dataset("tips")

@timer
def tip_rate_by_group(df, group_cols):
    """Compute mean tip rate (tip / total_bill) by the given grouping columns."""
    return (
        df.assign(tip_rate=df["tip"] / df["total_bill"])
          .groupby(group_cols, observed=True)["tip_rate"]
          .agg(["mean", "count"])
          .round(3)
    )

tip_rate_by_group(tips, ["day", "time"])

[INFO] @timer  tip_rate_by_group took 2.47 ms


mean  count
day  time                
Thur Lunch   0.161     61
     Dinner  0.160      1
Fri  Lunch   0.189      7
     Dinner  0.159     12
Sat  Dinner  0.153     87
Sun  Dinner  0.167     76

In [8]:
# Compare a vectorized operation vs. a Python loop on the same real data
@timer
def tip_rate_vectorized(df):
    """Vectorized tip-rate computation."""
    return (df["tip"] / df["total_bill"]).to_numpy()

@timer
def tip_rate_loop(df):
    """Row-by-row tip-rate computation (intentionally slow, for comparison)."""
    return np.array([row.tip / row.total_bill for row in df.itertuples()])

big_tips = pd.concat([tips] * 200, ignore_index=True)  # replicate real rows to make timing visible
v = tip_rate_vectorized(big_tips)
l = tip_rate_loop(big_tips)
print("Same result:", np.allclose(v, l))
print(f"Speed-up: {tip_rate_loop.last_elapsed / tip_rate_vectorized.last_elapsed:.0f}x")

[INFO] @timer  tip_rate_vectorized took 0.35 ms


[INFO] @timer  tip_rate_loop took 25.34 ms


Same result: True
Speed-up: 72x


## 1.4 Decorator `@log_call` — registrando chamadas e argumentos

Útil para **auditoria** de pipelines: quais funções foram chamadas, com quais parâmetros e
o que retornaram. Para DataFrames, registramos apenas o *shape* (e não o conteúdo inteiro!).

In [9]:
def _short_repr(obj, max_len=40):
    """Compact representation of an argument for logging purposes."""
    if isinstance(obj, pd.DataFrame):
        return f"DataFrame{obj.shape}"
    if isinstance(obj, pd.Series):
        return f"Series(len={len(obj)}, name={obj.name!r})"
    text = repr(obj)
    return text if len(text) <= max_len else text[: max_len - 3] + "..."


def log_call(func):
    """Log every call to `func` with its arguments and a short summary of the result."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        arg_txt = [_short_repr(a) for a in args]
        arg_txt += [f"{k}={_short_repr(v)}" for k, v in kwargs.items()]
        logger.info(f"@log_call -> {func.__name__}({', '.join(arg_txt)})")
        result = func(*args, **kwargs)
        logger.info(f"@log_call <- {func.__name__} returned {_short_repr(result)}")
        return result
    return wrapper

In [10]:
titanic = sns.load_dataset("titanic")

@log_call
def survival_rate(df, by="sex"):
    """Return the survival rate grouped by a categorical column."""
    return df.groupby(by, observed=True)["survived"].mean().round(3)

survival_rate(titanic, by="class")

[INFO] @log_call -> survival_rate(DataFrame(891, 15), by='class')


[INFO] @log_call <- survival_rate returned Series(len=3, name='survived')


class
First     0.630
Second    0.473
Third     0.242
Name: survived, dtype: float64

## 1.5 Decorator `@validate_dataframe` — validação de entradas

Um dos erros mais comuns em DS é processar um DataFrame vazio ou sem as colunas esperadas,
gerando resultados silenciosamente errados. Este decorator **falha cedo e com mensagem clara**.
Ele já recebe argumentos (`required_columns`), então é na verdade uma *fábrica de decorators* —
veremos esse padrão em detalhe na próxima seção.

In [11]:
def validate_dataframe(required_columns=(), allow_empty=False):
    """Decorator factory that validates the first positional argument is a proper DataFrame.

    Parameters
    ----------
    required_columns : iterable of str
        Columns that must be present in the DataFrame.
    allow_empty : bool
        If False, an empty DataFrame raises ValueError.
    """
    required = list(required_columns)

    def decorator(func):
        @functools.wraps(func)
        def wrapper(df, *args, **kwargs):
            if not isinstance(df, pd.DataFrame):
                raise TypeError(f"{func.__name__}: expected DataFrame, got {type(df).__name__}")
            if df.empty and not allow_empty:
                raise ValueError(f"{func.__name__}: received an empty DataFrame")
            missing = [c for c in required if c not in df.columns]
            if missing:
                raise ValueError(f"{func.__name__}: missing required columns {missing}")
            return func(df, *args, **kwargs)
        return wrapper
    return decorator

In [12]:
@validate_dataframe(required_columns=["age", "fare", "survived"])
def age_fare_summary(df):
    """Summarize age and fare for survivors vs. non-survivors."""
    return df.groupby("survived")[["age", "fare"]].median().round(1)

print(age_fare_summary(titanic))

# Failure cases are caught early with explicit messages
for bad_input in [titanic.iloc[0:0], titanic.drop(columns="fare"), titanic.to_dict()]:
    try:
        age_fare_summary(bad_input)
    except (ValueError, TypeError) as exc:
        print(f"{type(exc).__name__}: {exc}")

           age  fare
survived            
0         28.0  10.5
1         28.0  26.0
ValueError: age_fare_summary: received an empty DataFrame
ValueError: age_fare_summary: missing required columns ['fare']
TypeError: age_fare_summary: expected DataFrame, got dict


## 1.6 Decorators com argumentos (padrão *factory*): `@retry`

Quando o decorator precisa de parâmetros, criamos **três níveis** de funções:

1. `retry(times, delay)` → recebe a configuração e devolve o decorator;
2. `decorator(func)` → recebe a função;
3. `wrapper(*args, **kwargs)` → executa a lógica.

Cargas de dados vindas da rede (APIs, buckets, URLs) podem falhar de forma intermitente.
O `@retry` tenta novamente algumas vezes antes de desistir.

In [13]:
def retry(times=3, delay=0.1, exceptions=(Exception,)):
    """Retry `func` up to `times` attempts, sleeping `delay` seconds between attempts."""
    if times < 1:
        raise ValueError("times must be >= 1")

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    logger.warning(f"@retry  {func.__name__} attempt {attempt}/{times} failed: {exc}")
                    if attempt == times:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

In [14]:
# Simulate an unstable connection: the first 2 attempts fail, then the REAL dataset is loaded.
_attempts = {"count": 0}

@retry(times=3, delay=0.1, exceptions=(ConnectionError,))
def load_penguins_unstable():
    """Load the seaborn penguins dataset, failing on the first two calls (simulated network)."""
    _attempts["count"] += 1
    if _attempts["count"] < 3:
        raise ConnectionError("simulated timeout")
    return sns.load_dataset("penguins")

penguins = load_penguins_unstable()
print("Loaded penguins:", penguins.shape, "after", _attempts["count"], "attempts")

[WARNING] @retry  load_penguins_unstable attempt 1/3 failed: simulated timeout


[WARNING] @retry  load_penguins_unstable attempt 2/3 failed: simulated timeout


Loaded penguins: (344, 7) after 3 attempts


In [15]:
# When every attempt fails, the last exception is re-raised
@retry(times=2, delay=0.05, exceptions=(ConnectionError,))
def always_fails():
    """A loader that never succeeds."""
    raise ConnectionError("server unreachable")

try:
    always_fails()
except ConnectionError as exc:
    print("Gave up as expected ->", exc)

[WARNING] @retry  always_fails attempt 1/2 failed: server unreachable


[WARNING] @retry  always_fails attempt 2/2 failed: server unreachable


Gave up as expected -> server unreachable


## 1.7 `@functools.lru_cache` — memoização de cálculos caros

`lru_cache` guarda os resultados de chamadas anteriores (cache *Least Recently Used*).
Se a função for chamada de novo com os **mesmos argumentos**, o resultado vem da memória.

⚠️ Os argumentos precisam ser **hashable** — por isso passamos o *nome* do dataset (str),
e não o DataFrame (que é mutável e não-hashable). Atenção: o objeto retornado é compartilhado;
use `.copy()` se for modificá-lo.

In [16]:
@functools.lru_cache(maxsize=8)
def load_dataset_cached(name):
    """Load a seaborn dataset once and serve later calls from memory."""
    return sns.load_dataset(name)

for _ in range(3):
    t0 = time.perf_counter()
    df_cached = load_dataset_cached("titanic")
    print(f"titanic {df_cached.shape} loaded in {(time.perf_counter() - t0) * 1000:.3f} ms")

print(load_dataset_cached.cache_info())

titanic (891, 15) loaded in 4.125 ms
titanic (891, 15) loaded in 0.003 ms
titanic (891, 15) loaded in 0.002 ms
CacheInfo(hits=2, misses=1, maxsize=8, currsize=1)


In [17]:
# Classic example: naive recursive Fibonacci vs. memoized version
def fib_naive(n):
    """Exponential-time Fibonacci."""
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

@functools.lru_cache(maxsize=None)
def fib_cached(n):
    """Linear-time Fibonacci thanks to memoization."""
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)

for f in (fib_naive, fib_cached):
    t0 = time.perf_counter()
    value = f(27)
    print(f"{f.__name__:10s} fib(27)={value}  {(time.perf_counter() - t0) * 1000:8.2f} ms")
fib_cached.cache_clear()

fib_naive  fib(27)=196418     22.30 ms
fib_cached fib(27)=196418      0.01 ms


## 1.8 Empilhando múltiplos decorators

Decorators são aplicados **de baixo para cima** (o mais próximo da função é aplicado primeiro),
mas executados **de cima para baixo** na chamada:

```python
@timer          # 3º aplicado → camada mais externa
@log_call       # 2º aplicado
@validate_dataframe(...)  # 1º aplicado → camada mais interna
def f(df): ...
```

Ordem importa: aqui o `@timer` mede o tempo total, incluindo log e validação.

In [18]:
@timer
@log_call
@validate_dataframe(required_columns=["total_bill", "tip", "size"])
def tip_per_person(df):
    """Average tip per person, by party size."""
    return (df["tip"] / df["size"]).groupby(df["size"]).mean().round(2)

tip_per_person(tips)

[INFO] @log_call -> tip_per_person(DataFrame(244, 7))


[INFO] @log_call <- tip_per_person returned Series(len=6, name=None)


[INFO] @timer  tip_per_person took 3.49 ms


size
1    1.44
2    1.29
3    1.13
4    1.03
5    0.81
6    0.87
dtype: float64

In [19]:
# The stacked function still exposes the original metadata thanks to functools.wraps
print("name:", tip_per_person.__name__)
print("doc :", tip_per_person.__doc__)

# A failing validation still goes through @timer and @log_call (outer layers)
try:
    tip_per_person(tips.drop(columns="size"))
except ValueError as exc:
    print("Caught:", exc)

name: tip_per_person
doc : Average tip per person, by party size.
[INFO] @log_call -> tip_per_person(DataFrame(244, 6))


[INFO] @timer  tip_per_person took 1.03 ms


Caught: tip_per_person: missing required columns ['size']


## 1.9 Aplicando decorators a funções reais de DS (tips & titanic)

Vamos montar um pequeno conjunto de funções de limpeza/feature engineering para o Titanic,
todas instrumentadas com os decorators criados.

In [20]:
@timer
@validate_dataframe(required_columns=["age", "sex", "pclass"])
def impute_age(df):
    """Fill missing ages with the median age per (sex, pclass) group."""
    out = df.copy()
    out["age"] = out.groupby(["sex", "pclass"])["age"].transform(lambda s: s.fillna(s.median()))
    return out

@timer
@validate_dataframe(required_columns=["sibsp", "parch"])
def add_family_features(df):
    """Add family_size and is_alone features."""
    out = df.copy()
    out["family_size"] = out["sibsp"] + out["parch"] + 1
    out["is_alone"] = (out["family_size"] == 1).astype(int)
    return out

titanic_clean = add_family_features(impute_age(titanic))
print("Missing ages before/after:", titanic["age"].isna().sum(), "/", titanic_clean["age"].isna().sum())
titanic_clean.groupby("is_alone")["survived"].mean().round(3).rename("survival_rate")

[INFO] @timer  impute_age took 3.80 ms


[INFO] @timer  add_family_features took 0.87 ms


Missing ages before/after: 177 / 0


is_alone
0    0.506
1    0.304
Name: survival_rate, dtype: float64

In [21]:
@log_call
@validate_dataframe(required_columns=["total_bill", "tip", "smoker"])
def tip_stats_by_smoker(df):
    """Describe tip percentage for smokers vs. non-smokers."""
    pct = 100 * df["tip"] / df["total_bill"]
    return pct.groupby(df["smoker"], observed=True).describe()[["mean", "50%", "std"]].round(2)

tip_stats_by_smoker(tips)

[INFO] @log_call -> tip_stats_by_smoker(DataFrame(244, 7))


[INFO] @log_call <- tip_stats_by_smoker returned DataFrame(2, 3)


,mean,50%,std
smoker,,,
Yes,16.32,15.38,8.51
No,15.93,15.56,3.99


---
# Parte 2 — Context Managers

Um context manager implementa o **protocolo** `__enter__` / `__exit__`:

```python
with recurso() as r:   # __enter__ é chamado; seu retorno vai para `r`
    ...                # corpo do bloco
# __exit__ é chamado SEMPRE — com ou sem exceção
```

Se `__exit__` retornar `True`, a exceção é **suprimida**; se retornar `False`/`None`, ela é propagada.

## 2.1 Context manager baseado em classe: `DataPipelineContext`

In [22]:
class DataPipelineContext:
    """Track the lifecycle of a pipeline stage: timing, step log and error capture.

    Parameters
    ----------
    name : str
        Human-readable name of the pipeline.
    suppress_errors : bool
        If True, exceptions raised inside the block are logged and swallowed.
    """

    def __init__(self, name, suppress_errors=False):
        self.name = name
        self.suppress_errors = suppress_errors
        self.steps = []
        self.error = None
        self.elapsed = None

    def log_step(self, message):
        """Record a step with its timestamp relative to the pipeline start."""
        offset = time.perf_counter() - self._start
        self.steps.append((round(offset * 1000, 2), message))
        logger.info(f"[{self.name}] +{offset * 1000:.1f} ms  {message}")

    def __enter__(self):
        self._start = time.perf_counter()
        logger.info(f"[{self.name}] START")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self._start
        if exc_type is not None:
            self.error = exc_value
            logger.error(f"[{self.name}] FAILED after {self.elapsed * 1000:.1f} ms: "
                         f"{exc_type.__name__}: {exc_value}")
            return self.suppress_errors
        logger.info(f"[{self.name}] END ({self.elapsed * 1000:.1f} ms, {len(self.steps)} steps)")
        return False

    def summary(self):
        """Return the recorded steps as a DataFrame."""
        return pd.DataFrame(self.steps, columns=["offset_ms", "step"])

In [23]:
with DataPipelineContext("penguins-eda") as ctx:
    df_p = sns.load_dataset("penguins")
    ctx.log_step(f"loaded {df_p.shape}")
    df_p = df_p.dropna()
    ctx.log_step(f"dropped NA -> {df_p.shape}")
    body_mass = df_p.groupby("species")["body_mass_g"].mean().round(0)
    ctx.log_step("aggregated body mass by species")

print(body_mass)
ctx.summary()

[INFO] [penguins-eda] START


[INFO] [penguins-eda] +2.3 ms  loaded (344, 7)


[INFO] [penguins-eda] +3.2 ms  dropped NA -> (333, 7)


[INFO] [penguins-eda] +4.0 ms  aggregated body mass by species


[INFO] [penguins-eda] END (4.4 ms, 3 steps)


species
Adelie       3706.0
Chinstrap    3733.0
Gentoo       5092.0
Name: body_mass_g, dtype: float64


,offset_ms,step
0,2.29,"loaded (344, 7)"
1,3.17,"dropped NA -> (333, 7)"
2,4.02,aggregated body mass by species


In [24]:
# Error handling: with suppress_errors=True the exception is logged and stored, not raised
with DataPipelineContext("broken-stage", suppress_errors=True) as ctx_err:
    ctx_err.log_step("about to access a column that does not exist")
    _ = df_p["wingspan_cm"]  # KeyError

print("Pipeline continued. Stored error:", repr(ctx_err.error))

[INFO] [broken-stage] START


[INFO] [broken-stage] +1.2 ms  about to access a column that does not exist


[ERROR] [broken-stage] FAILED after 2.2 ms: KeyError: 'wingspan_cm'


Pipeline continued. Stored error: KeyError('wingspan_cm')


## 2.2 Context managers com `@contextlib.contextmanager`

Para casos simples, escrever uma classe é verboso. O decorator `contextlib.contextmanager`
transforma uma **função geradora** em context manager:

- o código **antes** do `yield` equivale ao `__enter__`;
- o valor do `yield` é o que vai para o `as`;
- o código no `finally` **depois** do `yield` equivale ao `__exit__`.

### `timer_context`

In [25]:
@contextlib.contextmanager
def timer_context(label):
    """Time the enclosed block; yields a dict that receives the elapsed seconds."""
    result = {"label": label, "elapsed": None}
    start = time.perf_counter()
    try:
        yield result
    finally:
        result["elapsed"] = time.perf_counter() - start
        logger.info(f"timer_context '{label}': {result['elapsed'] * 1000:.2f} ms")


with timer_context("titanic pivot") as t:
    pivot = titanic.pivot_table(index="class", columns="sex", values="survived",
                                aggfunc="mean", observed=True).round(3)
pivot

[INFO] timer_context 'titanic pivot': 3.88 ms


sex,female,male
class,,
First,0.968,0.369
Second,0.921,0.157
Third,0.500,0.135


### `temp_directory` — diretório temporário que se autodestrói

Ideal para artefatos intermediários (CSVs, modelos, gráficos) que não devem poluir o projeto.

In [26]:
@contextlib.contextmanager
def temp_directory(prefix="ds_tmp_"):
    """Create a temporary directory and remove it (with contents) on exit."""
    path = Path(tempfile.mkdtemp(prefix=prefix))
    try:
        yield path
    finally:
        shutil.rmtree(path, ignore_errors=True)
        logger.info(f"temp_directory removed: {path.name}")


with temp_directory() as tmp:
    csv_path = tmp / "tips_by_day.csv"
    tips.groupby("day", observed=True)[["total_bill", "tip"]].sum().to_csv(csv_path)
    print("Files inside:", [p.name for p in tmp.iterdir()])
    print(pd.read_csv(csv_path))

print("Directory still exists after block?", tmp.exists())

Files inside: ['tips_by_day.csv']
    day  total_bill     tip
0  Thur     1096.33  171.83
1   Fri      325.88   51.96
2   Sat     1778.40  260.40
3   Sun     1627.16  247.39


[INFO] temp_directory removed: ds_tmp_xcnjz2s3


Directory still exists after block? False


### `suppress_warnings` — silenciando avisos de forma controlada

Melhor do que desligar *todos* os warnings globalmente: silenciamos apenas uma categoria,
**apenas dentro do bloco**. `warnings.catch_warnings()` restaura o estado anterior ao sair.

In [27]:
@contextlib.contextmanager
def suppress_warnings(category=Warning):
    """Temporarily ignore warnings of the given category inside the block."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category)
        yield


def noisy_mean(s):
    """Compute a mean while emitting a (demonstration) warning."""
    warnings.warn("this is a demonstration warning", UserWarning)
    return s.mean()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    noisy_mean(tips["tip"])
print("Warnings without suppression:", len(caught))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    with suppress_warnings(UserWarning):
        value = noisy_mean(tips["tip"])
print("Warnings with suppression   :", len(caught), "| mean tip =", round(value, 3))

Warnings without suppression: 1
Warnings with suppression   : 0 | mean tip = 2.998


In [28]:
# The standard library also ships contextlib.suppress for ignoring specific exceptions
with contextlib.suppress(FileNotFoundError):
    os.remove("this_file_does_not_exist.csv")
print("No crash: FileNotFoundError was suppressed")

No crash: FileNotFoundError was suppressed


## 2.3 Context manager para operações seguras com arquivos

Escrever diretamente no arquivo final é arriscado: se o processo falhar no meio, ficamos com um
arquivo **corrompido/pela metade**. O padrão *atomic write* grava em um arquivo temporário e só
o renomeia para o destino se tudo der certo (`os.replace` é atômico no mesmo sistema de arquivos).

In [29]:
@contextlib.contextmanager
def atomic_write(target, mode="w", encoding="utf-8"):
    """Write to a temp file and atomically move it to `target` only on success."""
    target = Path(target)
    tmp_path = target.with_suffix(target.suffix + ".partial")
    fh = open(tmp_path, mode, encoding=encoding)
    try:
        yield fh
        fh.close()
        os.replace(tmp_path, target)
    except BaseException:
        fh.close()
        tmp_path.unlink(missing_ok=True)
        raise

In [30]:
with temp_directory() as tmp:
    target = tmp / "titanic_summary.csv"

    # 1) Successful write
    with atomic_write(target) as fh:
        titanic.groupby("class", observed=True)["fare"].describe().round(2).to_csv(fh)
    print("After success ->", [p.name for p in tmp.iterdir()])

    # 2) Failure mid-write: the previous good file is preserved, no partial file left
    try:
        with atomic_write(target) as fh:
            fh.write("class,broken\n")
            raise RuntimeError("crash while writing")
    except RuntimeError as exc:
        print("Write failed:", exc)
    print("After failure ->", [p.name for p in tmp.iterdir()])
    print(pd.read_csv(target).head(3))

After success -> ['titanic_summary.csv']
Write failed: crash while writing
After failure -> ['titanic_summary.csv']
    class  count   mean    std  min    25%    50%   75%     max
0   First  216.0  84.15  78.38  0.0  30.92  60.29  93.5  512.33
1  Second  184.0  20.66  13.42  0.0  13.00  14.25  26.0   73.50
2   Third  491.0  13.68  11.78  0.0   7.75   8.05  15.5   69.55
[INFO] temp_directory removed: ds_tmp_ejrdtb_g


## 2.4 Opções temporárias de exibição do pandas

O pandas já oferece `pd.option_context`, um context manager que altera opções de exibição
apenas dentro do bloco. Vamos usá-lo e também criar uma versão própria com *presets*.

In [31]:
print("Default precision:", pd.get_option("display.precision"))
with pd.option_context("display.precision", 1, "display.max_columns", 5):
    print(titanic[["age", "fare", "sibsp", "parch", "survived", "pclass"]].describe().loc[["mean", "std"]])
print("Restored precision:", pd.get_option("display.precision"))

Default precision: 6
       age  fare  ...  survived  pclass
mean  29.7  32.2  ...       0.4     2.3
std   14.5  49.7  ...       0.5     0.8

[2 rows x 6 columns]
Restored precision: 6


In [32]:
DISPLAY_PRESETS = {
    "compact": {"display.max_rows": 6, "display.max_columns": 6, "display.precision": 2},
    "wide": {"display.max_columns": None, "display.width": 250, "display.precision": 3},
}

@contextlib.contextmanager
def display_preset(name):
    """Apply a named group of pandas display options for the duration of the block."""
    options = DISPLAY_PRESETS[name]
    previous = {key: pd.get_option(key) for key in options}
    try:
        for key, val in options.items():
            pd.set_option(key, val)
        yield options
    finally:
        for key, val in previous.items():
            pd.set_option(key, val)

with display_preset("compact"):
    print(penguins)
print("max_rows restored to:", pd.get_option("display.max_rows"))

    species     island  bill_length_mm  ...  flipper_length_mm  body_mass_g  \
0    Adelie  Torgersen            39.1  ...              181.0       3750.0   
1    Adelie  Torgersen            39.5  ...              186.0       3800.0   
2    Adelie  Torgersen            40.3  ...              195.0       3250.0   
..      ...        ...             ...  ...                ...          ...   
341  Gentoo     Biscoe            50.4  ...              222.0       5750.0   
342  Gentoo     Biscoe            45.2  ...              212.0       5200.0   
343  Gentoo     Biscoe            49.9  ...              213.0       5400.0   

        sex  
0      Male  
1    Female  
2    Female  
..      ...  
341    Male  
342  Female  
343    Male  

[344 rows x 7 columns]
max_rows restored to: 60


## 2.5 Uso prático: conexões tipo banco de dados (sqlite3 em memória)

Conexões são o caso clássico de recurso que **precisa** ser fechado. Nosso context manager:

- abre a conexão;
- faz `commit` se o bloco terminar bem, ou `rollback` se ocorrer erro;
- **sempre** fecha a conexão.

(Atenção: `with sqlite3.connect(...) as conn` do Python faz commit/rollback, mas **não fecha** a conexão!)

In [33]:
@contextlib.contextmanager
def sqlite_connection(db_path=":memory:"):
    """Yield a sqlite3 connection with commit/rollback semantics and guaranteed close."""
    conn = sqlite3.connect(db_path)
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        logger.warning("sqlite_connection: transaction rolled back")
        raise
    finally:
        conn.close()
        logger.info("sqlite_connection: closed")


with sqlite_connection() as conn:
    tips.to_sql("tips", conn, index=False)
    query = """
        SELECT day, time,
               COUNT(*)                         AS n,
               ROUND(AVG(tip / total_bill), 3)  AS avg_tip_rate
        FROM tips
        GROUP BY day, time
        ORDER BY avg_tip_rate DESC
    """
    sql_result = pd.read_sql_query(query, conn)
sql_result

[INFO] sqlite_connection: closed


,day,time,n,avg_tip_rate
0,Fri,Lunch,7,0.189
1,Sun,Dinner,76,0.167
2,Thur,Lunch,61,0.161
3,Thur,Dinner,1,0.160
4,Fri,Dinner,12,0.159
5,Sat,Dinner,87,0.153


In [34]:
# Rollback demo: the INSERT inside the failing transaction is discarded
with temp_directory() as tmp:
    db = tmp / "demo.db"
    with sqlite_connection(db) as conn:
        titanic[["survived", "pclass", "sex", "age"]].to_sql("passengers", conn, index=False)

    try:
        with sqlite_connection(db) as conn:
            conn.execute("INSERT INTO passengers VALUES (1, 1, 'female', 30)")
            raise ValueError("validation failed after insert")
    except ValueError as exc:
        print("Error:", exc)

    with sqlite_connection(db) as conn:
        n_rows = conn.execute("SELECT COUNT(*) FROM passengers").fetchone()[0]
    print("Rows in table (should equal original 891):", n_rows)

[INFO] sqlite_connection: closed


[WARNING] sqlite_connection: transaction rolled back


[INFO] sqlite_connection: closed


Error: validation failed after insert
[INFO] sqlite_connection: closed


Rows in table (should equal original 891): 891
[INFO] temp_directory removed: ds_tmp_zem1mav5


---
# Parte 3 — Exemplos aplicados de DS

## 3.1 Mini pipeline completo: decorators + context managers

Vamos juntar tudo em um pipeline de pré-processamento do **iris**:

1. cada etapa é uma função pura decorada com `@timer`, `@log_call` e `@validate_dataframe`;
2. a execução completa roda dentro de `DataPipelineContext`;
3. as opções de exibição são controladas com `display_preset`.

In [35]:
FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

@timer
@log_call
def load_iris():
    """Load the seaborn iris dataset."""
    return sns.load_dataset("iris")

@timer
@log_call
@validate_dataframe(required_columns=FEATURES + ["species"])
def drop_duplicates_and_na(df):
    """Remove duplicated rows and rows with missing values."""
    return df.drop_duplicates().dropna().reset_index(drop=True)

@timer
@log_call
@validate_dataframe(required_columns=["petal_length", "petal_width", "sepal_length", "sepal_width"])
def add_ratio_features(df):
    """Add petal and sepal length/width ratio features."""
    out = df.copy()
    out["petal_ratio"] = out["petal_length"] / out["petal_width"]
    out["sepal_ratio"] = out["sepal_length"] / out["sepal_width"]
    return out

@timer
@log_call
@validate_dataframe(required_columns=FEATURES)
def standardize(df, columns=tuple(FEATURES)):
    """Z-score standardize the given numeric columns."""
    out = df.copy()
    cols = list(columns)
    out[cols] = (out[cols] - out[cols].mean()) / out[cols].std(ddof=0)
    return out

In [36]:
with DataPipelineContext("iris-preprocessing") as pipe:
    iris_raw = load_iris()
    pipe.log_step(f"raw shape {iris_raw.shape}")

    iris_clean = drop_duplicates_and_na(iris_raw)
    pipe.log_step(f"removed {len(iris_raw) - len(iris_clean)} duplicate/NA rows")

    iris_feat = add_ratio_features(iris_clean)
    pipe.log_step("added ratio features")

    iris_ready = standardize(iris_feat)
    pipe.log_step(f"standardized -> {iris_ready.shape}")

pipe.summary()

[INFO] [iris-preprocessing] START


[INFO] @log_call -> load_iris()


[INFO] @log_call <- load_iris returned DataFrame(150, 5)


[INFO] @timer  load_iris took 2.77 ms


[INFO] [iris-preprocessing] +3.9 ms  raw shape (150, 5)


...

[INFO] @log_call <- standardize returned DataFrame(149, 7)


[INFO] @timer  standardize took 5.21 ms


[INFO] [iris-preprocessing] +17.5 ms  standardized -> (149, 7)


[INFO] [iris-preprocessing] END (18.2 ms, 4 steps)


,offset_ms,step
0,3.90,"raw shape (150, 5)"
1,7.92,removed 1 duplicate/NA rows
2,11.54,added ratio features
3,17.48,"standardized -> (149, 7)"


In [37]:
# Sanity checks on the pipeline output
assert iris_ready[FEATURES].mean().abs().max() < 1e-9, "features should be centered"
assert np.allclose(iris_ready[FEATURES].std(ddof=0), 1.0), "features should have unit variance"

with display_preset("compact"):
    print(iris_ready.groupby("species")[["petal_ratio", "sepal_ratio"]].mean().round(2))
print(f"Total pipeline time: {pipe.elapsed * 1000:.1f} ms")

            petal_ratio  sepal_ratio
species                             
setosa             6.91         1.47
versicolor         3.24         2.16
virginica          2.78         2.23
Total pipeline time: 18.2 ms


In [38]:
# Same pipeline steps, but now validated on an independent source: sklearn's bundled iris
sk_iris = skdatasets.load_iris(as_frame=True)
sk_df = sk_iris.frame.rename(columns={
    "sepal length (cm)": "sepal_length", "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length", "petal width (cm)": "petal_width",
})
sk_df["species"] = sk_df["target"].map(dict(enumerate(sk_iris.target_names)))

with DataPipelineContext("sklearn-iris") as pipe2:
    out_sk = standardize(add_ratio_features(drop_duplicates_and_na(sk_df)))
    pipe2.log_step(f"done -> {out_sk.shape}")

# A broken input is stopped by the validator before any processing happens
with DataPipelineContext("broken-input", suppress_errors=True) as pipe3:
    standardize(sk_df.drop(columns="petal_width"))
print("Captured:", pipe3.error)

[INFO] [sklearn-iris] START


[INFO] @log_call -> drop_duplicates_and_na(DataFrame(150, 6))


[INFO] @log_call <- drop_duplicates_and_na returned DataFrame(149, 6)


[INFO] @timer  drop_duplicates_and_na took 1.31 ms


[INFO] @log_call -> add_ratio_features(DataFrame(149, 6))


...

[INFO] [broken-input] START


[INFO] @log_call -> standardize(DataFrame(150, 5))


[INFO] @timer  standardize took 0.35 ms


[ERROR] [broken-input] FAILED after 1.1 ms: ValueError: standardize: missing required columns ['petal_width']


Captured: standardize: missing required columns ['petal_width']


## 3.2 Ângulo NLP / Educação: decorators em processamento de texto

Como linguista e professor de inglês, me interessa analisar **vocabulário** e **complexidade
lexical** de textos. Usamos como corpus real os textos de documentação (`DESCR`) dos datasets
embutidos no scikit-learn — textos técnicos em inglês, públicos e distribuídos com a biblioteca.

Métricas usadas:
- **tokens** (total de palavras) e **types** (palavras distintas);
- **TTR** (*type-token ratio*) — diversidade lexical (sensível ao tamanho do texto!);
- **hapax legomena** — palavras que ocorrem uma única vez.

In [39]:
import re

loaders = {
    "iris": skdatasets.load_iris, "wine": skdatasets.load_wine,
    "breast_cancer": skdatasets.load_breast_cancer, "diabetes": skdatasets.load_diabetes,
    "digits": skdatasets.load_digits, "linnerud": skdatasets.load_linnerud,
}
corpus = {name: loader().DESCR for name, loader in loaders.items()}
print({name: len(text) for name, text in corpus.items()})
print(corpus["iris"][:300])

{'iris': 2656, 'wine': 3367, 'breast_cancer': 4794, 'diabetes': 1455, 'digits': 2007, 'linnerud': 704}
.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - pet


In [40]:
def validate_text(func):
    """Ensure the first argument is a non-empty string."""
    @functools.wraps(func)
    def wrapper(text, *args, **kwargs):
        if not isinstance(text, str):
            raise TypeError(f"{func.__name__}: expected str, got {type(text).__name__}")
        if not text.strip():
            raise ValueError(f"{func.__name__}: received empty text")
        return func(text, *args, **kwargs)
    return wrapper


TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")

@functools.lru_cache(maxsize=128)
@validate_text
def tokenize(text):
    """Lower-case alphabetic tokenization (cached; returns an immutable tuple)."""
    return tuple(TOKEN_RE.findall(text.lower()))

@validate_text
def word_count(text):
    """Number of alphabetic tokens in `text`."""
    return len(tokenize(text))

@validate_text
def vocabulary_profile(text, top_n=5):
    """Return lexical statistics: tokens, types, TTR, hapax ratio and top words (len > 3)."""
    tokens = tokenize(text)
    freq = Counter(tokens)
    hapax = sum(1 for c in freq.values() if c == 1)
    top = [w for w, _ in Counter(t for t in tokens if len(t) > 3).most_common(top_n)]
    return {
        "tokens": len(tokens),
        "types": len(freq),
        "ttr": round(len(freq) / len(tokens), 3),
        "hapax_ratio": round(hapax / len(freq), 3),
        "top_words": ", ".join(top),
    }

print("iris word count:", word_count(corpus["iris"]))
vocabulary_profile(corpus["iris"])

iris word count: 315


{'tokens': 315,
 'types': 181,
 'ttr': 0.575,
 'hapax_ratio': 0.68,
 'top_words': 'iris, class, each, data, classes'}

In [41]:
@timer
@log_call
def corpus_report(corpus_dict):
    """Build a vocabulary profile table for a {name: text} corpus."""
    rows = {name: vocabulary_profile(text) for name, text in corpus_dict.items()}
    return pd.DataFrame.from_dict(rows, orient="index").sort_values("tokens")

with timer_context("NLP corpus analysis"):
    report = corpus_report(corpus)
report

[INFO] @log_call -> corpus_report({'iris': '.. _iris_dataset:\n\nIris p...)


[INFO] @log_call <- corpus_report returned DataFrame(6, 5)


[INFO] @timer  corpus_report took 2.17 ms


[INFO] timer_context 'NLP corpus analysis': 2.39 ms


,tokens,types,ttr,hapax_ratio,top_words
linnerud,82,59,0.720,0.729,"dataset, exercise, physiological, variables, l..."
diabetes,196,130,0.663,0.685,"diabetes, blood, baseline, serum, each"
digits,276,167,0.605,0.707,"digits, recognition, handwritten, number, alpa..."
iris,315,181,0.575,0.680,"iris, class, each, data, classes"
wine,332,182,0.548,0.588,"class, wine, data, phenols, dept"
breast_cancer,474,216,0.456,0.625,"mean, worst, standard, error, radius"


In [42]:
# lru_cache on tokenize: repeated profiling reuses cached token tuples
tokenize.cache_clear()
for _ in range(3):
    for text in corpus.values():
        vocabulary_profile(text)
print(tokenize.cache_info())

# Text validator in action
for bad in ["   ", None]:
    try:
        word_count(bad)
    except (TypeError, ValueError) as exc:
        print(f"{type(exc).__name__}: {exc}")

CacheInfo(hits=12, misses=6, maxsize=128, currsize=6)
ValueError: word_count: received empty text
TypeError: word_count: expected str, got NoneType


In [43]:
# TTR is length-sensitive: compare on equal-size samples (first 300 tokens) for a fairer view
@validate_text
def standardized_ttr(text, n_tokens=300):
    """TTR computed on the first `n_tokens` tokens to control for text length."""
    tokens = tokenize(text)[:n_tokens]
    return round(len(set(tokens)) / len(tokens), 3)

report["ttr_300"] = [standardized_ttr(corpus[name]) for name in report.index]
report[["tokens", "ttr", "ttr_300"]].sort_values("ttr_300", ascending=False)

,tokens,ttr,ttr_300
linnerud,82,0.720,0.720
diabetes,196,0.663,0.663
digits,276,0.605,0.605
wine,332,0.548,0.597
iris,315,0.575,0.577
breast_cancer,474,0.456,0.470


**Leitura pedagógica:** o TTR bruto cai à medida que o texto cresce (as palavras funcionais se
repetem), por isso comparar textos de tamanhos diferentes exige padronização (`ttr_300`).
Em sala de aula, essa mesma lógica ajuda a escolher textos com diversidade lexical adequada ao
nível do aluno — e os decorators garantem que cada função de análise seja **validada, medida e
registrada** sem poluir a lógica linguística.

---
# Resumo — padrões aprendidos e quando usar cada um

| Padrão | O que faz | Quando usar em DS |
|---|---|---|
| Decorator básico + `functools.wraps` | Embrulha uma função preservando nome/docstring | Sempre que escrever um decorator |
| `@timer` | Mede tempo de execução | Encontrar gargalos, comparar vetorização vs. loops |
| `@log_call` | Registra chamadas, argumentos e retorno | Auditoria e depuração de pipelines |
| `@validate_dataframe` / `@validate_text` | Valida entradas e falha cedo | Funções que recebem dados externos ou de outras etapas |
| `@retry(times, delay)` (factory) | Repete operações instáveis | Carregamento via rede/API/bucket |
| `@functools.lru_cache` | Memoiza resultados | Cálculos caros e repetidos com argumentos *hashable* |
| Decorators empilhados | Combina comportamentos | Ordem importa: o de cima é a camada externa |
| Context manager de classe (`__enter__`/`__exit__`) | Estado + ciclo de vida completo | Pipelines com registro de etapas e tratamento de erro |
| `@contextlib.contextmanager` | Context manager via gerador | Casos simples: timers, diretórios temporários, warnings |
| `atomic_write` | Escrita segura de arquivos | Salvar resultados sem risco de arquivo corrompido |
| `pd.option_context` / `display_preset` | Opções temporárias do pandas | Relatórios e notebooks com formatação local |
| `sqlite_connection` | Commit/rollback + fechamento garantido | Qualquer conexão com banco de dados |

### Regras práticas
1. **Decorators** → comportamento *transversal* que se repete em várias funções (tempo, log, validação, cache, retry).
2. **Context managers** → recursos com *início e fim* obrigatórios (arquivos, conexões, configurações temporárias).
3. Use `try/finally` dentro de context managers geradores para garantir a limpeza.
4. Não exagere: um decorator deve deixar o código **mais claro**, não esconder lógica importante.

**Próximos passos:** aplicar esses padrões aos testes com pytest (Dia 10) e aos futuros pipelines de
EDA e NLP das próximas fases.